# Lab 07-02 — Faithfulness and correctness: two halves of answer quality

**Track 07 · Evaluation** — retrieval metrics stop at the context window; this lab scores what happens *after* generation: an answer can be factually right but unsupported by the retrieved context (hallucination), or supported but wrong against the reference answer (a retrieval gap the generator faithfully amplified).

This notebook is **self-contained**: it imports LangChain, sentence-transformers, faiss, and Groq directly — no repo component library. Every block of the pipeline is built right here: the BGE embedder, the FAISS index, the top-3 retriever, the answer generator, and the two scoring philosophies — an LLM-judge faithfulness rubric and the reference-based containment/cosine checks — all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

Two independent qualities:

* **Faithfulness** — does the answer follow ONLY from the retrieved context? A hand-rolled judge (same rubric and scale the lab uses) splits the answer into atomic claims and marks each supported/unsupported by the context; score = supported / total claims. This is the hallucination guard. It needs claim-rich answers — a terse "yes" has zero claims to check, which is why the generator is prompted to answer in complete sentences.
* **Correctness** — does the answer match the gold reference? Two reference-based measures: reference containment (the normalized gold answer appears inside the normalized answer — the only exact-style check that survives elaboration on a short-answer gold set) and embedding cosine between answer and reference (semantic tolerance for rephrasing).

The pipeline, drawn inline:

```text
rag-mini passages (3,200)
  -> BGE embed (HuggingFaceEmbeddings) + FAISS index
  -> retrieve top-3 per question (15 questions)
  -> ChatGroq generates the answer
  -> score: faithfulness (inline judge) + containment + cosine (reference)
  -> verification gate (--verify)
```


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/` (`passages.parquet` + `test.parquet`), already fetched by the repo's manifest-verified fetchers.
- **`GROQ_API_KEY` in the repo-root `.env`** — the generator and the judge both run on Groq's hosted Llama model; the imports cell loads the key via python-dotenv.

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-huggingface`, `langchain-community`, `langchain-groq`, `sentence-transformers`, `faiss-cpu`, `pandas`, and `python-dotenv`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`):
#   sentence-transformers -> local BGE embeddings (HuggingFaceEmbeddings)
#   langchain-huggingface -> the HuggingFaceEmbeddings wrapper
#   langchain-community   -> the FAISS vector store
#   faiss-cpu             -> the FAISS index
#   langchain-groq        -> ChatGroq (generator + judge)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
#   pandas                -> reads the passages/test.parquet corpus
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu langchain-groq python-dotenv pandas


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import pandas as pd  # noqa: E402  (reads the parquet corpus)

# LangChain + sentence-transformers + faiss + Groq — the only libraries this
# notebook needs. Nothing is imported from the repo's src/ component library.
from dotenv import load_dotenv  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_groq import ChatGroq  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)
load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `RAG_MINI` points at the rag-mini-wikipedia subset already on disk; `N_SAMPLE = 15` bounds the judge-evaluated questions (15 Groq generations + 15 judge calls — the LLM leg dominates the runtime); `TOP_K = 3` is the context fed to the generator; `BGE_MODEL_NAME` selects the local embedder; `GROQ_MODEL_NAME` selects the hosted generator/judge (the same model the lab's `GroqLLM` defaults to).


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# --------------------------------------------------------------------------
RAG_MINI = Path("Data/corpus/rag-mini-wikipedia")
PASSAGES_PATH = RAG_MINI / "passages.parquet"
TEST_PATH = RAG_MINI / "test.parquet"
N_SAMPLE = 15  # judge-evaluated questions (hosted LLM calls are the slow leg)
TOP_K = 3  # context chunks fed to the generator
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # local embedder (cached)
GROQ_MODEL_NAME = "llama-3.3-70b-versatile"  # generator + judge (lab default)


## 2. Load — passages + test QA

Two parquet files from rag-mini-wikipedia. Note the passages file has a **single `passage` column** — the loader reads it as plain text (no title/text fields to concatenate). The gold answers in `test.parquet` are terse — often just "yes", "no", or a number — which is why the correctness metrics below use *containment* and *cosine* instead of brittle exact-match.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — passages + test QA
# --------------------------------------------------------------------------
def load_passages(path: Path) -> tuple[list[str], list[str]]:
    """Return (doc_texts, doc_ids) for every passage."""
    df = pd.read_parquet(path)
    texts = [str(row["passage"]).strip() for _, row in df.iterrows()]
    ids = [str(i) for i in range(len(df))]
    return texts, ids


def load_test_qa(path: Path) -> list[dict]:
    """Return [{"question": ..., "answer": ...}] from test.parquet."""
    df = pd.read_parquet(path)
    return [{"question": r["question"], "answer": r["answer"]}
            for _, r in df.iterrows()]


## 3. Reference-based correctness (no LLM — deterministic)

The cheap, deterministic side of the evaluation: **cosine** between the answer embedding and the gold-reference embedding, and **containment** — is the gold text present in the answer? These need no LLM, are instant, and are perfectly reproducible. They are crude (they ignore wording that means the same thing), which is exactly why the next section adds an LLM judge on top.


In [ ]:
# --------------------------------------------------------------------------
# 3. Reference-based correctness (no LLM — deterministic)
# --------------------------------------------------------------------------
def normalize(text: str) -> str:
    """Lowercase, strip punctuation/articles and whitespace."""
    cleaned = "".join(c.lower() for c in text if c.isalnum() or c.isspace())
    words = [w for w in cleaned.split() if w not in ("a", "an", "the")]
    return " ".join(words)


def reference_contained(answer: str, reference: str) -> bool:
    """True when the normalized gold answer appears inside the answer.

    The rag-mini gold answers are terse ("yes", "no", a number) while the
    generator is asked to elaborate, so exact equality can never hold.
    Containment is the exact-style check that survives elaboration: the
    gold's normalized words must appear in the answer's normalized words.
    """
    return normalize(reference) in normalize(answer)


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Cosine similarity between two vectors (0.0 if either is zero)."""
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(x * x for x in b) ** 0.5
    if na == 0.0 or nb == 0.0:
        return 0.0
    return dot / (na * nb)


## 4. The judge + faithfulness metric — hand-rolled inline

The lab imports `LLMJudge` and `FaithfulnessMetric` from the shared evaluation block. Here both are built by hand with the **same rubric and scale**: `InlineJudge` wraps `ChatGroq` — `judge(instruction, prompt)` asks the model for a JSON object, strips a markdown code fence if the model wraps its answer, retries once on parse failure, and returns `{"error": ...}` when both attempts fail; `embed(texts)` returns local BGE vectors (the same model family the corpus was indexed with). `InlineFaithfulnessMetric.score` sends ONE judge call asking for `{"claims": [...], "supported": [bool, ...]}`, tolerates the singular `{"claim": ..., "supported": bool}` shape coder models often simplify to, and scores `supported / total` (0.0 when no claims parse). The rubric — and the failure mode it guards — is the point of the cell.


In [ ]:
# --------------------------------------------------------------------------
# 4. The judge + faithfulness metric — hand-rolled inline (same rubric)
# --------------------------------------------------------------------------
def _strip_code_fence(text: str) -> str:
    """Remove a surrounding markdown code fence (```json ... ```)."""
    lines = text.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


class InlineJudge:
    """LLM-as-judge over ChatGroq + local BGE embeddings (hand-rolled).

    Mirrors the shared LLMJudge contract the lab uses: ``judge()`` returns
    the parsed JSON dict (one retry on parse failure, then {"error": ...});
    ``embed()`` returns local BGE vectors.
    """

    def __init__(self, llm: ChatGroq, embedder):
        self.llm = llm
        self.embedder = embedder

    def judge(self, instruction: str, prompt: str) -> dict:
        last_error = ""
        for attempt in range(2):
            full = f"{instruction}\n\n{prompt}\n\nRespond with ONLY a valid JSON object."
            if attempt == 1:
                full += " Respond with ONLY valid JSON."
            try:
                text = self.llm.invoke(full).content
                return json.loads(_strip_code_fence(text))
            except (json.JSONDecodeError, ValueError) as exc:
                last_error = str(exc)
        return {"error": f"inline judge: could not parse JSON after 2 attempts: {last_error}"}

    def embed(self, texts: list[str]) -> list[list[float]]:
        return self.embedder.embed_documents(texts)


class InlineFaithfulnessMetric:
    """Faithfulness: fraction of the answer's claims supported by context.

    Same rubric as the shared FaithfulnessMetric: one judge call asking for
    ``{"claims": [...], "supported": [bool, ...]}``; score = supported /
    total claims (0.0 when no claims parse).
    """

    def __init__(self, judge):
        self.judge = judge

    def score(self, question: str, context: str, answer: str) -> float:
        instruction = (
            "You are a faithfulness judge. Break the answer into atomic "
            "claims, then mark each claim as supported (true) or not (false) "
            "by the context."
        )
        prompt = (
            f"Context:\n{context}\n\nAnswer:\n{answer}\n\n"
            'Return JSON: {"claims": ["..."], "supported": [true, false, ...]}'
        )
        result = self.judge.judge(instruction, prompt)
        if "error" in result:
            return 0.0
        claims, supported = self._normalize(result)
        n = min(len(claims), len(supported))
        if n == 0:
            return 0.0
        return sum(1 for flag in supported[:n] if flag) / n

    @staticmethod
    def _normalize(result: dict) -> tuple[list[str], list[bool]]:
        """Coerce the judge's JSON into parallel (claims, supported) lists.

        The judge is asked for ``{"claims": [...], "supported": [...]}`` but
        coder models routinely simplify to a singular ``{"claim": "...",
        "supported": true}``. Accept both: read ``claims``/``claim`` and
        ``supported`` as a list OR a single bool (a scalar bool is applied to
        every claim).
        """
        claims = result.get("claims")
        if not claims:
            claim = result.get("claim")
            claims = [claim] if claim else []
        supported = result.get("supported")
        if isinstance(supported, bool):
            supported = [supported] * len(claims)
        elif not isinstance(supported, list):
            supported = []
        return list(claims), list(supported)


## 5. Experiment — retrieve, generate, score

The full pipeline: embed all 3,200 passages with the local BGE model (normalized — BGE requires it for cosine), index them in FAISS through a precomputed-vector passthrough so the embed step and the index step stay separately timed, retrieve the top-3 context per question, let ChatGroq generate an answer, then score it three ways — **faithfulness** (the inline judge splits the answer into claims and checks each against the context), **containment**, and **cosine**.

`device="cpu"` on the embedder mirrors the lab: bulk-embedding 3,200 passages on CPU keeps the run light on the shared machine's GPU. The thread cap below keeps the BLAS/OpenMP footprint small while several track agents share this box.


In [ ]:
# --------------------------------------------------------------------------
# 5. Experiment — retrieve, generate, score
# --------------------------------------------------------------------------
# Several track agents share this machine — cap BLAS/OpenMP threads so the
# BGE embedding step stays light on CPU and memory.
os.environ["OMP_NUM_THREADS"] = "2"
import torch  # noqa: E402
torch.set_num_threads(2)


class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call. embed_query is delegated to the real embedder so
    the store's retriever can embed queries.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]],
                 query_embedder: Embeddings):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))
        self._query_embedder = query_embedder

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._query_embedder.embed_query(text)


def run_experiment() -> dict:
    passages, passage_ids = load_passages(PASSAGES_PATH)
    test_qa = load_test_qa(TEST_PATH)

    # --- Embed locally (BGE, CPU) and index in-memory -----------------------
    embedder = HuggingFaceEmbeddings(
        model_name=BGE_MODEL_NAME,
        model_kwargs={"device": "cpu"},  # leave the shared GPU alone
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    t0 = time.perf_counter()
    vectors = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    chunks = [
        Document(page_content=t, metadata={"id": cid})
        for t, cid in zip(passages, passage_ids)
    ]
    t0 = time.perf_counter()
    store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings(passages, vectors, embedder)
    )
    index_s = time.perf_counter() - t0
    retriever = store.as_retriever(search_kwargs={"k": TOP_K})

    # --- Generator + hand-rolled judge (both ChatGroq, temp 0) --------------
    llm = ChatGroq(model=GROQ_MODEL_NAME, temperature=0.0)
    judge = InlineJudge(llm, embedder)
    faithfulness = InlineFaithfulnessMetric(judge)

    rows: list[dict] = []
    t0 = time.perf_counter()
    for item in test_qa[:N_SAMPLE]:
        question, reference = item["question"], item["answer"]
        context_docs = retriever.invoke(question)
        context = "\n\n".join(d.page_content for d in context_docs)
        answer = llm.invoke(
            f"Context:\n{context}\n\nQuestion: {question}\n\n"
            "Answer in one or two complete sentences, stating the key "
            "fact(s) from the context:"
        ).content.strip()

        rows.append({
            "question": question,
            "reference": reference,
            "answer": answer,
            "faithfulness": faithfulness.score(question, context, answer),
            "contained": reference_contained(answer, reference),
            "cosine": cosine_similarity(
                judge.embed([answer])[0], judge.embed([reference])[0]
            ),
        })
    llm_s = time.perf_counter() - t0

    def mean(key: str) -> float:
        vals = [r[key] for r in rows]
        return sum(vals) / len(vals) if vals else 0.0

    return {
        "rows": rows,
        "indexed": len(passages),
        "embed_s": embed_s,
        "index_s": index_s,
        "llm_s": llm_s,
        "metrics": {
            "faithfulness": mean("faithfulness"),
            "containment": mean("contained"),
            "answer_cosine": mean("cosine"),
        },
    }


## 6. Demo — print the artifact

`print_demo(exp)` prints the aggregate scores plus three example rows. Watch the *relationship* between faithfulness and containment: an answer can be perfectly faithful to an irrelevant context (high faithfulness, low containment) — the generator faithfully amplified a retrieval miss. That combination is how you localize the fault to retrieval rather than the LLM.


In [ ]:
# --------------------------------------------------------------------------
# 6. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 07-02 — Faithfulness and correctness")
    print(f"rag-mini {exp['indexed']} passages, {len(exp['rows'])} questions")
    print("=" * 66)

    m = exp["metrics"]
    print(f"\n[1] Mean scores over {len(exp['rows'])} questions:")
    print(f"    faithfulness (claims supported by context): {m['faithfulness']:.3f}")
    print(f"    reference containment                     : {m['containment']:.3f}")
    print(f"    answer-reference cosine                   : {m['answer_cosine']:.3f}")

    print("\n[2] Three example rows:")
    for row in exp["rows"][:3]:
        print(f"    Q: {row['question'][:60]}")
        print(f"       gold={row['reference'][:40]!r} | "
              f"faith={row['faithfulness']:.2f} | "
              f"contained={row['contained']} | cos={row['cosine']:.2f}")

    print(f"\n[3] Timing: embed {exp['indexed']} passages {exp['embed_s']:.1f}s, "
          f"index {exp['index_s']:.1f}s, LLM+judge {len(exp['rows'])} Q {exp['llm_s']:.1f}s")

    print(f"\n[4] Takeaway")
    print("    Faithfulness and correctness are orthogonal: an answer can be")
    print("    perfectly faithful to a context that is itself irrelevant")
    print("    (high faithfulness, low containment), or wrong while fully")
    print("    supported by the evidence (the generator faithfully amplified")
    print("    a retrieval miss). Two operational notes: faithfulness needs")
    print("    claim-rich answers — a terse 'yes' has zero claims and scores")
    print("    0.0 by construction — and short-answer gold sets force you to")
    print("    soften exact-match into containment or cosine. High faithfulness")
    print("    + low correctness points at retrieval, not the LLM.")


## 7. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: `N_SAMPLE` questions evaluated, faithfulness and containment in [0, 1], cosine in [-1, 1], and — the non-trivial one — **faithfulness > 0**: elaborated answers carry claims; a terse "yes" would score 0.0 by construction, which is why the prompt asks for complete sentences. This is the same gate the CI-style `--verify` run applies; every check should print PASS.


In [ ]:
# --------------------------------------------------------------------------
# 7. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    m = exp["metrics"]
    n = len(exp["rows"])

    checks.append((f"{N_SAMPLE} questions evaluated (>= 10)", n >= 10))
    checks.append(("faithfulness mean in [0, 1]", 0.0 <= m["faithfulness"] <= 1.0))
    checks.append(("faithfulness > 0 (elaborated answers carry claims)",
                   m["faithfulness"] > 0.0))
    checks.append(("reference containment in [0, 1]",
                   0.0 <= m["containment"] <= 1.0))
    checks.append(("answer cosine in [-1, 1]", -1.0 <= m["answer_cosine"] <= 1.0))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Bulk-embedding 3,200 passages on CPU takes a few minutes (no downloads — the BGE model is cached); the 15 ChatGroq generations + judge calls follow. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The three-way score: mean faithfulness (claims supported by context), reference containment, and answer-reference cosine over the 15 questions, plus three example rows. Watch the faithfulness/containment relationship — an answer can be faithful to an irrelevant context.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the rag-mini files are intact and `GROQ_API_KEY` is set.


In [ ]:
verify_gate(exp)
